# GEOG 592 — Module 7 Homework
## GeoPandas Practice



#### 1. Complete the hierarchy:
1. Shapely geometry
2. GeoSeries
3. GeoDataframe
#### 2. What type of object is returned by each expression? Write Shapely geometry, GeoSeries, GeoDataFrame, or pandas Series.
parks.geometry\
GeoSeries\
parks.geometry.iloc[0]\
Shapely geometry\
parks.iloc[0]\
pandas Series\
parks.iloc[[0]]\
GeoDataFrame
#### 3. Explain in one sentence why `parks.geometry.iloc[0]` does not behave exactly like `parks.geometry`.
parks.geometry.iloc[0] is very specifically only one geometry whereas parks.geometry contains a series of geometries.
#### 4. Write code to display all column names in `parks` as a regular Python list.
```python
parks.columns.tolist()
```
#### 5. Write code to select only the `NAME` and `geometry` columns from `parks`.
```python
parks[["Name", "geometry"]]
```
#### 6. Write code to select the first three rows of `parks` while keeping the result as a GeoDataFrame.
```python
parks.iloc[:3]
```
#### 7. Write code to inspect the geometry type of every feature in `parks`.
```python
parks.geometry.geom_type
```
#### 8. What does `parks.crs` tell you?
parks.crs will tell us which coordinate reference system (CRS) that the data is stored in because differences can change where the data is actually on Earth.
#### 9. Why can one point still produce a complete graph when you call `.plot()`?
.plot() will create a figure with axes using Matplotlib which can automaticaly create the surroundings of only one point using the data that is already present in Matplotlib.
#### 10. Assume `one_park = parks.geometry.iloc[0]`. What library provides the geometry object stored in `one_park`?
Shapely
```python
stops.crs
# EPSG:4326
```
#### 11. Would EPSG:4326 be a good CRS for measuring distance in feet? Explain briefly.
No because EPSG.4326 is measured in degrees so measurements in feet are not possible. Instead you would have to reproject the data with a different CRS that store units in feet.
#### 12. Write code to create `stops_nc` by reprojecting `stops` to EPSG:2264.
```python
stops_nc = stops.to_crs(epsg=2264)
```
#### 13. Write code to create an `area_sqft` column in `parks_nc` using polygon geometry area.
```python
parks_nc["area_sqrt"] = parks_nc.geometry.area
```
#### 14. Write code to calculate the distance from every stop in `stops_nc` to a single polygon named `park_1`.
```python
stops_nc.geometry.distance(park_1)
```
#### 15. If `park_1` is a polygon, does `.distance(park_1)` measure distance to the polygon centroid? Explain.
No the distance method will calculate the minimum distance between the geometry and the polygon instead of the distance to it centroid. This might mean it would measure the distance from the outer edge of park_1 rather than the center.
#### 16. Explain each
within: The entire left geometry is inside the right geometry\
contains: The left geometry contains the right geometry\
intersects: Two geometries have some shared point\
touches: The boundaries of two geometries touch but the interior does not overlap
#### 17. In `gpd.sjoin(parks, urban, predicate="within")`, which geometry is being tested as within which?
Testing which parks are within urban
#### 18. A park crosses an urban-area boundary. Would `within` be True? Would `intersects` be True?
Within would not be true becuase it has to hold the entirety of the park. However, intersects is true because intersects only requires for parks of the geometry to be within the other.
#### 19. Write code that returns how many stops are inside each polygon in a gpd named `parks` that has three polygons.
```python
joined = gpd.sjoin(stops, parks, predicate="within")

counts = (
    joined.groupby("index_right")
    .size()
    .reindex(parks.index, fill_value=0)
)
```
```python
urban_parks = gpd.sjoin(parks, urban, predicate="within")
```
#### 20. What is the purpose of `gpd.sjoin()`?
gdp.sjoin() will do a spatial join by combining each row from difference GeoDataFrames using different spatial relationships, like 'within' above.
#### 21. Why is a spatial join different from a regular pandas join on a common ID field?
A spatial join will macth a row based on relationship between geometries while a regular pandas join just matches rows that have the same value.
#### 22. Write a spatial join that keeps bus stops that fall inside park polygons. Use `stops` as the left GeoDataFrame and `parks` as the right GeoDataFrame.
```python
stops_parks = gpd.sjoin(
    stops,
    parks,
    how="inner",
    predicate="within"
)
```
#### 23. After a point-in-polygon spatial join, how could you count how many bus stops were matched to each park? Describe the idea or write code.
```python
stops_in_parks.groupby("index_right").size()
```
#### 24. Why can a polygon-to-polygon `intersects` spatial join create multiple output rows if each original layer was one polygon in size?
A spatial join will make make only one row for each matching geometry. So if a polygon was intersecting with multiple polygons in another GeoDataFrame, the polygon would appear multiple times.
#### 25. You want to answer: “Which parks are within 500 feet of a bus stop?” Write how would you do this with GeoPandas
### a. using a distance and apply method.
```python
within_500 = parks_nc[
    parks_nc.geometry.apply(
        lambda park: stops_nc.distance(park).min() <=
    )
]
```
### b. using buffers
```python
buffers = stops_nc.copy()
buffers["geometries"] = stops_nc.buffer(500)

nearby = gpd.sjoin(
    parks_nc,
    stop_buffers,
    predicate="intersects"
)
```